In [1]:
# ============================================================
# Qwen2.5 FAIR 4-WAY CONTROLLED COMPARISON
# Google Colab one-cell script — v2 GPU/pandas-safe
#
# Systems:
#   1) BASE_RAW
#   2) BASE_FEMSEG_MAPP
#   3) FT_RAW
#   4) FT_FEMSEG_MAPP
#
# Fairness controls:
#   - one exact-QA-deduplicated dataset
#   - one leakage-safe GROUPED train/test split by normalized question
#   - same train candidate IDs and same test IDs for every system
#   - same Qwen checkpoint and 4-bit quantization for every system
#   - same max lengths and dense retrieval rule
#   - same fixed external answer-semantic evaluator for every system
#   - no morphology reranker in the controlled comparison
#   - no answer post-processing in the controlled comparison
#   - fine-tuning objective matches retrieval: question -> question contrastive learning
#   - fresh identical base initialization for FT_RAW and FT_FEMSEG_MAPP
#   - paired bootstrap CIs and paired tests on the same test items
# ============================================================

# --- Colab GPU preflight: stop BEFORE downloads if GPU is not enabled ---
import sys, subprocess
import torch as _torch_preflight

if not _torch_preflight.cuda.is_available():
    print("\n[STOP] CUDA GPU анықталмады.")
    print("Google Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    print("Содан кейін Save/Connect жасап, осы cell-ді қайта іске қосыңыз.")
    raise RuntimeError("GPU қажет: current runtime = CPU")

print("GPU:", _torch_preflight.cuda.get_device_name(0))
print("CUDA:", _torch_preflight.version.cuda)

# Keep Colab's required pandas version; do NOT use -U pandas.
_packages = [
    "transformers", "accelerate", "peft", "bitsandbytes",
    "sentence-transformers", "bert-score", "scikit-learn", "scipy",
    "pandas==2.2.3", "pytorch-crf"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_packages])

del _torch_preflight

import os, re, gc, glob, json, time, random, hashlib, unicodedata
from pathlib import Path
from typing import List, Dict, Any, Optional, Set, Tuple
from collections import Counter

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sentence_transformers import SentenceTransformer
from scipy.stats import wilcoxon, binomtest

from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from bert_score import score as bert_score

try:
    from torchcrf import CRF
except Exception as e:
    raise ImportError(
        "torchcrf импортталмады. Colab Runtime -> Restart session жасап, осы cell-ді қайта іске қосыңыз."
    ) from e

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# 1) REPRODUCIBILITY / HARDWARE
# ============================================================
SEED = 42

def set_all_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("GPU жоғалып кетті. Colab Runtime -> Change runtime type -> T4 GPU таңдаңыз.")
print("GPU NAME:", torch.cuda.get_device_name(0))

# ============================================================
# 2) CONFIG — CHANGE ONLY PATHS IF NEEDED
# ============================================================
DATA_JSON_PATH = "grok_python_dataset.json"

FEMSEG_CHAR2ID = "/content/drive/MyDrive/KAZ_MORPH/char2id_femseg_v3_50k.json"
FEMSEG_CKPT = "/content/drive/MyDrive/KAZ_MORPH/femseg_v3_50k_epoch3.pt"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = True                    # SAME for all 4 systems
TEST_SIZE = 0.10
TOPK = 3

# Same token budgets for raw and morphology-aware systems.
MAX_QUERY_LEN = 256
MAX_CANDIDATE_LEN = 256
EVAL_BATCH_SIZE = 8
SIM_BATCH_SIZE = 256

# Fine-tuning — SAME for FT_RAW and FT_FEMSEG_MAPP
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
TEMPERATURE = 0.05
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Independent semantic evaluator: SAME frozen model for all systems.
FIXED_SEM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SEM_THR = 0.85
SEM_BATCH_SIZE = 64

RUN_BERTSCORE = True
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
BERTSCORE_BATCH_SIZE = 16

# Paired inference
BOOT_N = 2000
BOOT_SEED = 2026

# Output
OUTPUT_DIR = "/content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results" if IN_COLAB else "./fair_qwen_compare_results"

# MAPP
USE_MAPP = True
MAPP_TAG = "[MAPP]"
FEMSEG_TAG = "[FEMSEG]"
MAPP_MIN_WORD_LEN = 4

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)

# ============================================================
# 3) ROBUST JSON LOADING
# ============================================================
def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p
    candidates = [
        f"/content/{p}",
        f"/content/drive/MyDrive/{p}",
        f"/content/drive/MyDrive/KAZ_MORPH/{p}",
    ]
    for c in candidates:
        if Path(c).exists():
            return c
    name = Path(p).name
    hits = glob.glob(f"/content/**/{name}", recursive=True) if Path("/content").exists() else []
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Dataset табылмады: {p}")


def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)


def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)


def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text


def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None
    if "question" in rec and "answer" in rec:
        q, a = str(rec["question"]).strip(), str(rec["answer"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    if "instruction" in rec and "response" in rec:
        q, a = str(rec["instruction"]).strip(), str(rec["response"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    return None


def _extract_records(obj: Any) -> List[Dict[str, str]]:
    one = _normalize_record(obj)
    if one is not None:
        return [one]
    if isinstance(obj, list):
        return [x for x in (_normalize_record(v) for v in obj) if x is not None]
    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if isinstance(obj.get(key), list):
                return [x for x in (_normalize_record(v) for v in obj[key]) if x is not None]
    return []


def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    # single JSON
    try:
        out = _extract_records(json.loads(_normalize_json_text(text)))
        if out:
            return out
    except Exception:
        pass
    # concatenated JSON objects
    try:
        t = _normalize_json_text(text)
        dec, i, out = json.JSONDecoder(), 0, []
        while i < len(t):
            while i < len(t) and t[i] in " \r\n\t,":
                i += 1
            if i >= len(t):
                break
            obj, j = dec.raw_decode(t, i)
            i = j
            out.extend(_extract_records(obj))
        if out:
            return out
    except Exception:
        pass
    # JSONL
    out = []
    for line in text.splitlines():
        s = line.strip().rstrip(",")
        if not s:
            continue
        try:
            out.extend(_extract_records(json.loads(_normalize_json_text(s))))
        except Exception:
            continue
    if not out:
        raise ValueError("JSON ішінен QA жазбалар оқылмады.")
    return out

# ============================================================
# 4) TEXT NORMALIZATION + METRICS
# ============================================================
_punct_space_left = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space = re.compile(r"\s+")


def clean_text(text: str) -> str:
    t = "" if text is None else str(text)
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def norm_for_group(text: str) -> str:
    t = clean_text(text).lower()
    # Group near-identical questions together: punctuation/spacing/case do not create leakage.
    t = re.sub(r"[^a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9_+#]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_text(text).lower()).strip()


def tokens(text: str) -> List[str]:
    t = clean_text(text).lower()
    return re.findall(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+", t)


def token_f1(pred: str, gold: str) -> float:
    p, g = tokens(pred), tokens(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    inter = sum((pc & gc).values())
    if inter == 0:
        return 0.0
    precision = inter / len(p)
    recall = inter / len(g)
    return 2 * precision * recall / (precision + recall + 1e-12)

# ============================================================
# 5) FEMSeg — SAME IMPLEMENTATION AS YOUR NOTEBOOK
# ============================================================
BMES_TAGS = ["B", "M", "E", "S"]
TAG2ID = {t: i for i, t in enumerate(BMES_TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}


class FEMSegV3(nn.Module):
    def __init__(self, vocab_size: int, char_emb_dim: int = 128,
                 lstm_hidden_dim: int = 256, dropout: float = 0.3,
                 num_tags: int = len(BMES_TAGS)):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(char_emb_dim, lstm_hidden_dim, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward_features(self, x, mask):
        emb = self.dropout(self.emb(x))
        h, _ = self.lstm(emb)
        return self.dropout(h)

    def forward_logits(self, x, mask):
        return self.fc(self.forward_features(x, mask))

    def forward(self, x, tags=None, mask=None):
        if mask is None:
            mask = x != 0
        emissions = self.forward_logits(x, mask)
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)


def bmes_to_cse(chars: List[str], tags: List[str]) -> str:
    morphs, cur = [], ""
    for ch, t in zip(chars, tags):
        if t == "B":
            if cur:
                morphs.append(cur)
            cur = ch
        elif t == "M":
            cur += ch
        elif t == "E":
            cur += ch
            morphs.append(cur)
            cur = ""
        elif t == "S":
            if cur:
                morphs.append(cur)
            morphs.append(ch)
            cur = ""
    if cur:
        morphs.append(cur)
    return "@@ ".join(morphs)


class KazMorphSegmentor:
    def __init__(self, char2id_path: str, ckpt_path: str, use_cuda: bool = True):
        with open(char2id_path, "r", encoding="utf-8") as f:
            self.char2id = json.load(f)
        self.pad_id = 0
        self.unk_id = self.char2id.get("<unk>") or self.char2id.get("[UNK]") or 1
        self.device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")
        vocab_size = max(self.char2id.values()) + 1
        self.model = FEMSegV3(vocab_size=vocab_size).to(self.device)
        state = torch.load(ckpt_path, map_location=self.device)
        if isinstance(state, dict):
            if "state_dict" in state:
                state = state["state_dict"]
            elif "model_state_dict" in state:
                state = state["model_state_dict"]
        if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        self.word_cache, self.text_cache = {}, {}

    def _word_to_tensor(self, word: str):
        ids = [self.char2id.get(ch, self.unk_id) for ch in list(word)]
        x = torch.tensor([ids], dtype=torch.long, device=self.device)
        return x, x != self.pad_id

    def segment_word(self, word: str) -> str:
        word = word.strip()
        if not word:
            return ""
        if word in self.word_cache:
            return self.word_cache[word]
        x, mask = self._word_to_tensor(word)
        with torch.no_grad():
            best_paths = self.model(x, mask=mask)
        tags = [ID2TAG[i] for i in best_paths[0]]
        result = bmes_to_cse(list(word), tags)
        self.word_cache[word] = result
        return result

    def segment_text(self, text: str) -> str:
        text = text.strip()
        if not text:
            return ""
        if text in self.text_cache:
            return self.text_cache[text]
        result = " | ".join(self.segment_word(w) for w in text.split())
        self.text_cache[text] = result
        return result


def femseg_to_sp(text: str) -> str:
    text = clean_text(str(text))
    if not text:
        return ""
    seg_line = femseg.segment_text(text)
    sp_tokens = []
    for word_seg in seg_line.split(" | "):
        morphs = [m.strip() for m in word_seg.strip().split("@@ ") if m.strip()]
        if not morphs:
            continue
        sp_tokens.append("▁" + morphs[0])
        sp_tokens.extend(morphs[1:])
    return " ".join(sp_tokens)

# ============================================================
# 6) MAPP — SAME CORE LOGIC AS YOUR NOTEBOOK
# ============================================================
WORD_RE = re.compile(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9_+#-]+", re.UNICODE)

PROTECTED_WORDS = {
    "не", "неге", "неліктен", "қалай", "қайда", "қайдан", "қашан",
    "қандай", "қанша", "қай", "кім", "кімге", "кімнің", "неше",
    "python", "def", "for", "while", "if", "elif", "else", "print", "input",
    "list", "dict", "tuple", "set", "int", "str", "float", "bool", "none",
    "true", "false", "class", "return", "break", "continue", "range", "len",
    "append", "insert", "remove", "pop", "sort", "lambda", "map", "filter",
    "reduce", "zip", "enumerate", "and", "or", "not", "in", "is",
}

MAPP_WORD_MAP = {
    "калай": "қалай", "кандай": "қандай", "кайда": "қайда", "кашан": "қашан",
    "канша": "қанша", "аныкта": "анықта", "аныктайды": "анықтайды",
    "аныктаймыз": "анықтаймыз", "есептеймз": "есептейміз", "пайтон": "python",
    "питон": "python", "функсия": "функция", "функсиа": "функция",
    "циклд": "цикл", "стр": "str", "инт": "int", "бул": "bool",
    "принт": "print", "инпут": "input",
}

_mapp_word_cache, _mapp_text_cache = {}, {}


def _basic_mapp_word_norm(word: str) -> str:
    w = str(word).strip().lower()
    return MAPP_WORD_MAP.get(w, w)


def _is_ascii_programming_token(word: str) -> bool:
    return bool(re.fullmatch(r"[a-z0-9_+#-]+", word))


def _split_seg_word(seg_word: str) -> List[str]:
    return [m.strip() for m in str(seg_word).split("@@ ") if m.strip()]


def _mapp_stem_from_femseg(word: str) -> str:
    w = _basic_mapp_word_norm(word)
    if not USE_MAPP or not w:
        return w
    if w in _mapp_word_cache:
        return _mapp_word_cache[w]
    if w in PROTECTED_WORDS or len(w) < MAPP_MIN_WORD_LEN or w.isdigit() or _is_ascii_programming_token(w):
        _mapp_word_cache[w] = w
        return w
    try:
        morphs = _split_seg_word(femseg.segment_word(w))
        if not morphs:
            result = w
        else:
            stem = _basic_mapp_word_norm(morphs[0])
            result = stem if len(stem) >= 2 else w
    except Exception:
        result = w
    _mapp_word_cache[w] = result
    return result


def mapp_normalize(text: str) -> str:
    t = clean_text(text).lower().strip()
    if not t:
        return ""
    if t in _mapp_text_cache:
        return _mapp_text_cache[t]
    words = WORD_RE.findall(t)
    out, prev = [], None
    for w in words:
        nw = _mapp_stem_from_femseg(w)
        if nw and nw != prev:
            out.append(nw)
            prev = nw
    result = " ".join(out).strip() or t
    _mapp_text_cache[t] = result
    return result


def compose_morph_view(q_clean: str, q_sp: str, q_mapp: str) -> str:
    q_clean, q_sp, q_mapp = clean_text(q_clean), clean_text(q_sp), clean_text(q_mapp)
    parts = [q_clean] if q_clean else []
    if q_sp:
        parts.extend([FEMSEG_TAG, q_sp])
    if q_mapp and norm_for_exact(q_mapp) != norm_for_exact(q_clean):
        parts.extend([MAPP_TAG, q_mapp])
    return " ".join(parts).strip()


# ============================================================
# 7) LOAD, DEDUPLICATE, GROUP-SPLIT — ONCE ONLY
# ============================================================
data_path = find_data_path(DATA_JSON_PATH)
print("\nDataset:", data_path)
raw_records = load_qa_json_robust(data_path)
qa = pd.DataFrame(raw_records)
qa["q_clean"] = qa["question"].map(clean_text)
qa["a_clean"] = qa["answer"].map(clean_text)
qa = qa[(qa.q_clean != "") & (qa.a_clean != "")].reset_index(drop=True)
qa["q_group"] = qa["q_clean"].map(norm_for_group)
qa["a_norm"] = qa["a_clean"].map(norm_for_exact)

n_before = len(qa)
qa = qa.drop_duplicates(subset=["q_group", "a_norm"], keep="first").reset_index(drop=True)
qa["row_id"] = np.arange(len(qa), dtype=np.int64)
print(f"Rows before exact-QA dedup: {n_before}")
print(f"Rows after exact-QA dedup : {len(qa)}")
print(f"Unique normalized questions: {qa.q_group.nunique()}")

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(qa, groups=qa["q_group"]))
train_df = qa.iloc[train_idx].copy().reset_index(drop=True)
test_df = qa.iloc[test_idx].copy().reset_index(drop=True)

train_groups = set(train_df.q_group)
test_groups = set(test_df.q_group)
overlap = train_groups & test_groups
assert len(overlap) == 0, f"LEAKAGE: {len(overlap)} question groups overlap!"

print(f"TRAIN rows: {len(train_df)} | groups: {train_df.q_group.nunique()}")
print(f"TEST  rows: {len(test_df)} | groups: {test_df.q_group.nunique()}")
print("Question-group overlap:", len(overlap))
print("Exact answer overlap across split:", len(set(train_df.a_norm) & set(test_df.a_norm)))

# Save split manifest before any modeling.
split_manifest = pd.concat([
    train_df[["row_id", "q_group"]].assign(split="train"),
    test_df[["row_id", "q_group"]].assign(split="test"),
], ignore_index=True)
split_manifest.to_csv(Path(OUTPUT_DIR) / "split_manifest.csv", index=False)

# ============================================================
# 8) FEMSeg + MAPP VIEWS — DETERMINISTIC, NO FITTING ON TEST
# ============================================================
assert Path(FEMSEG_CHAR2ID).exists(), f"FEMSeg char2id табылмады: {FEMSEG_CHAR2ID}"
assert Path(FEMSEG_CKPT).exists(), f"FEMSeg checkpoint табылмады: {FEMSEG_CKPT}"

print("\nFEMSeg model жүктелуде...")
femseg = KazMorphSegmentor(FEMSEG_CHAR2ID, FEMSEG_CKPT, use_cuda=True)
print("FEMSeg дайын.")

# Build maps on all unique strings only for speed. This is deterministic inference, not fitting.
all_unique_q = pd.concat([train_df.q_clean, test_df.q_clean]).drop_duplicates().tolist()
q_sp_map, q_mapp_map = {}, {}
print("\nFEMSeg/MAPP preprocessing...")
t0 = time.time()
for i, qtxt in enumerate(all_unique_q, start=1):
    q_sp_map[qtxt] = femseg_to_sp(qtxt)
    q_mapp_map[qtxt] = mapp_normalize(qtxt)
    if i % 1000 == 0 or i == len(all_unique_q):
        print(f"  {i}/{len(all_unique_q)} | {time.time()-t0:.1f}s")

for df in (train_df, test_df):
    df["q_sp"] = df.q_clean.map(q_sp_map)
    df["q_mapp"] = df.q_clean.map(q_mapp_map)
    df["q_morph"] = [compose_morph_view(a,b,c) for a,b,c in zip(df.q_clean, df.q_sp, df.q_mapp)]

print("\nView example:")
display(train_df[["q_clean", "q_morph", "a_clean"]].head(2))

# Free FEMSeg GPU memory before Qwen.
try:
    femseg.model.to("cpu")
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 9) FIXED EXTERNAL ANSWER SEMANTIC EVALUATOR — PRECOMPUTE ONCE
# ============================================================
print("\nFixed semantic evaluator:", FIXED_SEM_MODEL)
sem_model = SentenceTransformer(FIXED_SEM_MODEL, device="cuda")
train_answer_sem = sem_model.encode(
    train_df.a_clean.tolist(), batch_size=SEM_BATCH_SIZE,
    convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype(np.float32)
test_answer_sem = sem_model.encode(
    test_df.a_clean.tolist(), batch_size=SEM_BATCH_SIZE,
    convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype(np.float32)
del sem_model
gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 10) QWEN ENCODER HELPERS
# ============================================================
print("\nTokenizer:", QWEN_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)


def load_base_qwen():
    set_all_seeds(SEED)
    model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config if USE_4BIT else None,
        torch_dtype=None if USE_4BIT else torch.float16,
        device_map="auto" if USE_4BIT else None,
        trust_remote_code=True,
    )
    if not USE_4BIT:
        model = model.to(DEVICE)
    model.config.use_cache = False
    model.eval()
    return model


def load_fresh_lora_qwen():
    # Fresh identical base + fresh adapter initialization for each FT condition.
    set_all_seeds(SEED)
    model = AutoModel.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config if USE_4BIT else None,
        torch_dtype=None if USE_4BIT else torch.float16,
        device_map="auto" if USE_4BIT else None,
        trust_remote_code=True,
    )
    if not USE_4BIT:
        model = model.to(DEVICE)
    model.config.use_cache = False
    if USE_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    try:
        model.enable_input_require_grads()
    except Exception:
        pass
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    model = get_peft_model(model, lora_cfg)
    model.config.use_cache = False
    try:
        model.print_trainable_parameters()
    except Exception:
        pass
    return model


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def encode_batch_with_grad(model, texts: List[str], max_length: int) -> torch.Tensor:
    dev = next(model.parameters()).device
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_length,
        return_tensors="pt"
    )
    batch = {k: v.to(dev) for k, v in batch.items()}
    out = model(**batch)
    emb = mean_pool(out.last_hidden_state, batch["attention_mask"])
    return F.normalize(emb.float(), p=2, dim=1)


@torch.inference_mode()
def encode_texts(model, texts: List[str], max_length: int, batch_size: int = EVAL_BATCH_SIZE) -> np.ndarray:
    model.eval()
    chunks = []
    dev = next(model.parameters()).device
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start+batch_size]
        batch = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=max_length,
            return_tensors="pt"
        )
        batch = {k: v.to(dev) for k, v in batch.items()}
        out = model(**batch)
        emb = mean_pool(out.last_hidden_state, batch["attention_mask"])
        emb = F.normalize(emb.float(), p=2, dim=1)
        chunks.append(emb.cpu().numpy().astype(np.float32))
    return np.vstack(chunks)

# ============================================================
# 11) RETRIEVAL-MATCHED CONTRASTIVE FINE-TUNING
# ============================================================
class QuestionDataset(Dataset):
    def __init__(self, questions: List[str]):
        self.questions = questions
    def __len__(self):
        return len(self.questions)
    def __getitem__(self, idx):
        return self.questions[idx]


def collate_questions(batch):
    return list(batch)


def symmetric_infonce(z1: torch.Tensor, z2: torch.Tensor, temperature: float) -> torch.Tensor:
    logits = (z1 @ z2.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def finetune_condition(question_col: str, system_name: str):
    # One representative per normalized question avoids same-question false negatives.
    # Two forward passes of the SAME question are positive pairs. LoRA dropout supplies
    # stochastic views, while other questions in the batch are negatives. This directly
    # optimizes the same question-to-question retrieval used at evaluation.
    ft_df = train_df.drop_duplicates(subset=["q_group"], keep="first").reset_index(drop=True)
    ds = QuestionDataset(ft_df[question_col].tolist())
    gen = torch.Generator()
    gen.manual_seed(SEED)
    loader = DataLoader(
        ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=gen,
        num_workers=0, collate_fn=collate_questions, drop_last=True
    )

    model = load_fresh_lora_qwen()
    model.train()

    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    updates_per_epoch = max(1, int(np.ceil(len(loader) / GRAD_ACCUM_STEPS)))
    total_steps = updates_per_epoch * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)

    print(f"\n===== TRAIN {system_name} =====")
    print("FT unique question groups:", len(ft_df))
    print("Micro batch:", TRAIN_BATCH_SIZE, "| grad accum:", GRAD_ACCUM_STEPS,
          "| effective batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
    print("Total optimizer steps:", total_steps)

    opt.zero_grad(set_to_none=True)
    global_step = 0
    for epoch in range(NUM_EPOCHS):
        running = 0.0
        for bi, q_texts in enumerate(loader):
            # Independent LoRA-dropout passes create two stochastic views of each question.
            z1 = encode_batch_with_grad(model, q_texts, MAX_QUERY_LEN)
            z2 = encode_batch_with_grad(model, q_texts, MAX_QUERY_LEN)
            loss = symmetric_infonce(z1, z2, TEMPERATURE) / GRAD_ACCUM_STEPS
            loss.backward()
            running += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

            do_step = ((bi + 1) % GRAD_ACCUM_STEPS == 0) or (bi + 1 == len(loader))
            if do_step:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM
                )
                opt.step(); scheduler.step(); opt.zero_grad(set_to_none=True)
                global_step += 1

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | mean microbatch loss={running/max(1,len(loader)):.6f}")

    model.eval()
    return model

# ============================================================
# 12) DENSE RETRIEVAL — IDENTICAL RULE FOR ALL SYSTEMS
# ============================================================
def retrieve_topk(query_emb: np.ndarray, doc_emb: np.ndarray, k: int = TOPK) -> Tuple[np.ndarray, np.ndarray]:
    n = query_emb.shape[0]
    k = min(k, doc_emb.shape[0])
    all_idx, all_sim = [], []
    for start in range(0, n, SIM_BATCH_SIZE):
        qb = query_emb[start:start+SIM_BATCH_SIZE]
        sims = qb @ doc_emb.T
        # argpartition then exact descending order inside top-k
        idx = np.argpartition(-sims, kth=k-1, axis=1)[:, :k]
        vals = np.take_along_axis(sims, idx, axis=1)
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        vals = np.take_along_axis(vals, order, axis=1)
        all_idx.append(idx.astype(np.int32))
        all_sim.append(vals.astype(np.float32))
    return np.vstack(all_idx), np.vstack(all_sim)


def evaluate_system(model, system_name: str, question_col: str) -> Dict[str, Any]:
    print(f"\n===== EVALUATE {system_name} =====")
    t0 = time.time()
    train_q_emb = encode_texts(model, train_df[question_col].tolist(), MAX_CANDIDATE_LEN)
    test_query_emb = encode_texts(model, test_df[question_col].tolist(), MAX_QUERY_LEN)
    topk_idx, topk_sim = retrieve_topk(test_query_emb, train_q_emb, TOPK)

    gold = test_df.a_clean.tolist()
    top1_idx = topk_idx[:, 0]
    pred = train_df.a_clean.iloc[top1_idx].tolist()
    topk_answers = [[train_df.a_clean.iloc[j] for j in row] for row in topk_idx]

    # Standard lexical/exact metrics
    exact = np.array([float(norm_for_exact(p) == norm_for_exact(g)) for p,g in zip(pred,gold)], dtype=np.float32)
    top3_exact = np.array([
        float(any(norm_for_exact(a) == norm_for_exact(g) for a in row))
        for row,g in zip(topk_answers,gold)
    ], dtype=np.float32)
    tokf1 = np.array([token_f1(p,g) for p,g in zip(pred,gold)], dtype=np.float32)
    qsim = topk_sim[:,0].astype(np.float32)

    # FIXED evaluator embeddings — never use the evaluated Qwen model here.
    pred_sem = train_answer_sem[top1_idx]
    ans_cos = np.sum(pred_sem * test_answer_sem, axis=1).astype(np.float32)
    topk_sem = train_answer_sem[topk_idx]  # [N,K,D]
    topk_ans_cos = np.sum(topk_sem * test_answer_sem[:,None,:], axis=2).astype(np.float32)
    sem1 = (ans_cos >= SEM_THR).astype(np.float32)
    recall3 = np.any(topk_ans_cos >= SEM_THR, axis=1).astype(np.float32)
    mrr3 = []
    for row in (topk_ans_cos >= SEM_THR):
        rr = 0.0
        for rank, hit in enumerate(row, start=1):
            if hit:
                rr = 1.0 / rank
                break
        mrr3.append(rr)
    mrr3 = np.asarray(mrr3, dtype=np.float32)

    out = {
        "system": system_name,
        "question_col": question_col,
        "topk_idx": topk_idx,
        "topk_sim": topk_sim,
        "pred_answers": pred,
        "gold_answers": gold,
        "topk_answers": topk_answers,
        "Exact@1": exact,
        "Top3ExactHit": top3_exact,
        "TokenF1@1": tokf1,
        "MeanCos@1(QSim)": qsim,
        "AnswerCos@1_FIXED": ans_cos,
        "Semantic@1_FIXED": sem1,
        "Recall@3_FIXED": recall3,
        "MRR@3_FIXED": mrr3,
    }
    print(f"Done {system_name} | {time.time()-t0:.1f}s")
    return out

# ============================================================
# 13) RUN FOUR CONTROLLED SYSTEMS
# ============================================================
systems: Dict[str, Dict[str, Any]] = {}

# A. BASE — same exact frozen model instance for RAW and MORPH.
print("\nLoading frozen BASE Qwen once for both base conditions...")
base_model = load_base_qwen()
systems["BASE_RAW"] = evaluate_system(base_model, "BASE_RAW", "q_clean")
systems["BASE_FEMSEG_MAPP"] = evaluate_system(base_model, "BASE_FEMSEG_MAPP", "q_morph")
del base_model
gc.collect(); torch.cuda.empty_cache()

# B. FT_RAW — fresh base + fresh LoRA.
ft_raw_model = finetune_condition("q_clean", "FT_RAW")
systems["FT_RAW"] = evaluate_system(ft_raw_model, "FT_RAW", "q_clean")
del ft_raw_model
gc.collect(); torch.cuda.empty_cache()

# C. FT_MORPH — fresh SAME base + fresh SAME LoRA config/seed.
ft_morph_model = finetune_condition("q_morph", "FT_FEMSEG_MAPP")
systems["FT_FEMSEG_MAPP"] = evaluate_system(ft_morph_model, "FT_FEMSEG_MAPP", "q_morph")
del ft_morph_model
gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 14) BERTSCORE — ONE FIXED EVALUATOR CALL FOR ALL SYSTEMS
# ============================================================
if RUN_BERTSCORE:
    print("\nComputing BERTScore for all systems with ONE fixed evaluator...")
    names = list(systems.keys())
    all_preds, all_gold, spans = [], [], {}
    cursor = 0
    for name in names:
        p = systems[name]["pred_answers"]
        g = systems[name]["gold_answers"]
        all_preds.extend(p); all_gold.extend(g)
        spans[name] = (cursor, cursor + len(p)); cursor += len(p)
    _, _, f1 = bert_score(
        all_preds, all_gold,
        model_type=BERTSCORE_MODEL_TYPE,
        lang="kk",
        batch_size=BERTSCORE_BATCH_SIZE,
        rescale_with_baseline=False,
        device="cuda",
        verbose=True,
    )
    f1 = f1.detach().cpu().numpy().astype(np.float32)
    for name, (a,b) in spans.items():
        systems[name]["BERTScoreF1@1"] = f1[a:b]
else:
    for name in systems:
        systems[name]["BERTScoreF1@1"] = np.full(len(test_df), np.nan, dtype=np.float32)

gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 15) SUMMARY TABLE
# ============================================================
metric_names = [
    "Exact@1", "Top3ExactHit", "TokenF1@1", "MeanCos@1(QSim)",
    "AnswerCos@1_FIXED", "Semantic@1_FIXED", "Recall@3_FIXED", "MRR@3_FIXED",
    "BERTScoreF1@1",
]

summary_rows = []
for name, res in systems.items():
    row = {"System": name}
    for m in metric_names:
        row[m] = float(np.nanmean(res[m]))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

print("\n================ FAIR 4-WAY SUMMARY ================")
display(summary_df)
summary_df.to_csv(Path(OUTPUT_DIR) / "fair_4way_summary.csv", index=False)

# ============================================================
# 16) ITEMWISE EXPORT — SAME TEST ITEM ORDER FOR ALL SYSTEMS
# ============================================================
itemwise = test_df[["row_id", "question", "answer", "q_group"]].copy()
for name, res in systems.items():
    itemwise[f"{name}__pred"] = res["pred_answers"]
    itemwise[f"{name}__top3"] = [json.dumps(x, ensure_ascii=False) for x in res["topk_answers"]]
    for m in metric_names:
        itemwise[f"{name}__{m}"] = res[m]
itemwise.to_csv(Path(OUTPUT_DIR) / "fair_4way_itemwise.csv", index=False)

# ============================================================
# 17) PAIRED BOOTSTRAP + PAIRED TESTS
# ============================================================
def paired_bootstrap_diff(a: np.ndarray, b: np.ndarray, n_boot: int = BOOT_N, seed: int = BOOT_SEED):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    d = b[mask] - a[mask]
    if len(d) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boots = d[idx].mean(axis=1)
    return float(d.mean()), float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))


def paired_pvalue(a: np.ndarray, b: np.ndarray, binary: bool):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if binary:
        # Exact McNemar = exact binomial on discordant pairs.
        b01 = int(np.sum((a == 0) & (b == 1)))
        b10 = int(np.sum((a == 1) & (b == 0)))
        n = b01 + b10
        p = 1.0 if n == 0 else float(binomtest(min(b01,b10), n=n, p=0.5, alternative="two-sided").pvalue)
        return p, f"Exact McNemar (discordant={n})"
    d = b - a
    if np.allclose(d, 0):
        return 1.0, "Wilcoxon signed-rank"
    try:
        p = float(wilcoxon(a, b, zero_method="pratt", alternative="two-sided", method="auto").pvalue)
    except Exception:
        nz = d[np.abs(d) > 1e-12]
        if len(nz) == 0:
            p = 1.0
        else:
            pos = int(np.sum(nz > 0))
            p = float(binomtest(pos, n=len(nz), p=0.5, alternative="two-sided").pvalue)
        return p, "Exact sign test fallback"
    return p, "Wilcoxon signed-rank"

comparisons = [
    ("BASE_RAW", "BASE_FEMSEG_MAPP", "Morphology effect without FT"),
    ("FT_RAW", "FT_FEMSEG_MAPP", "Morphology effect with FT"),
    ("BASE_RAW", "FT_RAW", "Fine-tuning effect on RAW"),
    ("BASE_FEMSEG_MAPP", "FT_FEMSEG_MAPP", "Fine-tuning effect on MORPH"),
]

binary_metrics = {"Exact@1", "Top3ExactHit", "Semantic@1_FIXED", "Recall@3_FIXED"}
stats_rows = []
for a_name, b_name, label in comparisons:
    for m in metric_names:
        a, b = systems[a_name][m], systems[b_name][m]
        diff, lo, hi = paired_bootstrap_diff(a, b)
        p, test_name = paired_pvalue(a, b, binary=(m in binary_metrics))
        stats_rows.append({
            "Comparison": label,
            "A": a_name,
            "B": b_name,
            "Metric": m,
            "Mean_A": float(np.nanmean(a)),
            "Mean_B": float(np.nanmean(b)),
            "Delta_B_minus_A": diff,
            "Bootstrap95CI_low": lo,
            "Bootstrap95CI_high": hi,
            "p_value_two_sided": p,
            "paired_test": test_name,
        })

stats_df = pd.DataFrame(stats_rows)
print("\n================ PAIRED STATISTICS ================")
display(stats_df)
stats_df.to_csv(Path(OUTPUT_DIR) / "fair_4way_paired_statistics.csv", index=False)

# ============================================================
# 18) PROTOCOL / VALIDITY CHECK REPORT
# ============================================================
protocol = {
    "seed": SEED,
    "dataset_path": data_path,
    "rows_after_exact_qa_dedup": int(len(qa)),
    "unique_question_groups": int(qa.q_group.nunique()),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_question_groups": int(train_df.q_group.nunique()),
    "test_question_groups": int(test_df.q_group.nunique()),
    "question_group_overlap": int(len(overlap)),
    "qwen_model": QWEN_MODEL_NAME,
    "same_4bit_all_systems": bool(USE_4BIT),
    "retrieval": "test question -> fixed train question pool; cosine; dense only; top-k=3; selected train answer returned",
    "max_query_len": MAX_QUERY_LEN,
    "max_candidate_len": MAX_CANDIDATE_LEN,
    "morph_rerank_used": False,
    "answer_postprocessing_used": False,
    "fixed_semantic_evaluator": FIXED_SEM_MODEL,
    "semantic_threshold": SEM_THR,
    "bertscore_model": BERTSCORE_MODEL_TYPE if RUN_BERTSCORE else None,
    "ft_objective": "symmetric in-batch InfoNCE on two stochastic LoRA-dropout passes of the same training question; aligned to question-to-question retrieval",
    "ft_epochs": NUM_EPOCHS,
    "ft_micro_batch": TRAIN_BATCH_SIZE,
    "ft_grad_accum": GRAD_ACCUM_STEPS,
    "ft_lr": LEARNING_RATE,
    "ft_temperature": TEMPERATURE,
}
with open(Path(OUTPUT_DIR) / "fair_4way_protocol.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)

print("\n================ VALIDITY CHECK ================")
print("[OK] Same grouped split for all systems")
print("[OK] Question-group leakage = 0")
print("[OK] Same candidate row IDs for all systems")
print("[OK] Same Qwen checkpoint / 4-bit precision")
print("[OK] Same query/candidate-question token budgets")
print("[OK] Same dense cosine retrieval; no morph reranker")
print("[OK] Same raw candidate answers; no answer post-processing")
print("[OK] Fixed external semantic evaluator for all systems")
print("[OK] FT objective matches question-to-question retrieval target")
print("[OK] FT_RAW and FT_MORPH start from fresh identical base/LoRA seed")
print("[OK] Paired statistics use the exact same test items")
print("\nSaved:")
for fn in [
    "split_manifest.csv",
    "fair_4way_summary.csv",
    "fair_4way_itemwise.csv",
    "fair_4way_paired_statistics.csv",
    "fair_4way_protocol.json",
]:
    print(" -", str(Path(OUTPUT_DIR) / fn))

print("\nIMPORTANT INTERPRETATION:")
print("1) BASE_RAW vs BASE_FEMSEG_MAPP = morphology effect without fine-tuning.")
print("2) FT_RAW vs FT_FEMSEG_MAPP = morphology effect under the same fine-tuning regime.")
print("3) Do NOT attribute any change here to reranking/post-processing: they are intentionally absent.")
print("4) Semantic@1/Recall@3/MRR@3 use the SAME fixed MiniLM evaluator, not each Qwen system itself.")
print("5) If SEM_THR=0.85 is to be claimed as a validated decision threshold in a paper, calibrate it on a separate validation set, never on this test set.")

GPU: Tesla T4
CUDA: 12.8
DEVICE: cuda
GPU NAME: Tesla T4
Mounted at /content/drive
OUTPUT_DIR: /content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results

Dataset: grok_python_dataset.json
Rows before exact-QA dedup: 5944
Rows after exact-QA dedup : 5725
Unique normalized questions: 5463
TRAIN rows: 5153 | groups: 4916
TEST  rows: 572 | groups: 547
Question-group overlap: 0
Exact answer overlap across split: 26

FEMSeg model жүктелуде...
FEMSeg дайын.

FEMSeg/MAPP preprocessing...
  1000/5537 | 2.3s
  2000/5537 | 3.8s
  3000/5537 | 5.3s
  4000/5537 | 6.4s
  5000/5537 | 7.7s
  5537/5537 | 8.1s

View example:


,q_clean,q_morph,a_clean
0,Python дегеніміз не?,Python дегеніміз не? [FEMSEG] ▁Python ▁де ген ...,Python-1991 жылы Гвидо ван Россум жасаған қара...
1,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты қ...,Python бағдарламалау тілінің негізгі мақсаты-а...



Fixed semantic evaluator: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

done in 4.52 seconds, 506.50 sentences/sec

================ FAIR 4-WAY SUMMARY ================


,System,Exact@1,Top3ExactHit,TokenF1@1,MeanCos@1(QSim),AnswerCos@1_FIXED,Semantic@1_FIXED,Recall@3_FIXED,MRR@3_FIXED,BERTScoreF1@1
0,BASE_RAW,0.006993,0.015734,0.351777,0.992964,0.558458,0.162587,0.269231,0.210373,0.780429
1,BASE_FEMSEG_MAPP,0.008741,0.013986,0.354409,0.994609,0.560674,0.180070,0.274476,0.223485,0.784203
2,FT_RAW,0.005245,0.017483,0.357553,0.663771,0.564897,0.188811,0.288462,0.231643,0.782428
3,FT_FEMSEG_MAPP,0.006993,0.017483,0.369002,0.696150,0.573285,0.181818,0.270979,0.218823,0.789667



================ PAIRED STATISTICS ================


,Comparison,A,B,Metric,Mean_A,Mean_B,Delta_B_minus_A,Bootstrap95CI_low,Bootstrap95CI_high,p_value_two_sided,paired_test
0,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,Exact@1,0.006993,0.008741,0.001748,0.000000,0.005245,1.000000e+00,Exact McNemar (discordant=1)
1,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,Top3ExactHit,0.015734,0.013986,-0.001748,-0.008741,0.003497,1.000000e+00,Exact McNemar (discordant=3)
2,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,TokenF1@1,0.351777,0.354409,0.002632,-0.009111,0.013681,2.258423e-01,Wilcoxon signed-rank
3,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,MeanCos@1(QSim),0.992964,0.994609,0.001645,0.001447,0.001870,3.485652e-53,Wilcoxon signed-rank
4,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,AnswerCos@1_FIXED,0.558458,0.560674,0.002216,-0.010015,0.014459,8.933142e-01,Wilcoxon signed-rank
5,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,Semantic@1_FIXED,0.162587,0.180070,0.017483,0.001748,0.034965,6.391466e-02,Exact McNemar (discordant=24)
6,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,Recall@3_FIXED,0.269231,0.274476,0.005245,-0.013986,0.026224,7.358788e-01,Exact McNemar (discordant=35)
7,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,MRR@3_FIXED,0.210373,0.223485,0.013112,-0.000291,0.027098,1.502942e-01,Wilcoxon signed-rank
8,Morphology effect without FT,BASE_RAW,BASE_FEMSEG_MAPP,BERTScoreF1@1,0.780429,0.784203,0.003774,-0.000965,0.008800,1.172437e-01,Wilcoxon signed-rank
9,Morphology effect with FT,FT_RAW,FT_FEMSEG_MAPP,Exact@1,0.005245,0.006993,0.001748,0.000000,0.005245,1.000000e+00,Exact McNemar (discordant=1)



================ VALIDITY CHECK ================
[OK] Same grouped split for all systems
[OK] Question-group leakage = 0
[OK] Same candidate row IDs for all systems
[OK] Same Qwen checkpoint / 4-bit precision
[OK] Same query/candidate-question token budgets
[OK] Same dense cosine retrieval; no morph reranker
[OK] Same raw candidate answers; no answer post-processing
[OK] Fixed external semantic evaluator for all systems
[OK] FT objective matches question-to-question retrieval target
[OK] FT_RAW and FT_MORPH start from fresh identical base/LoRA seed
[OK] Paired statistics use the exact same test items

Saved:
 - /content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results/split_manifest.csv
 - /content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results/fair_4way_summary.csv
 - /content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results/fair_4way_itemwise.csv
 - /content/drive/MyDrive/KAZ_MORPH/fair_qwen_compare_results/fair_4way_paired_statistics.csv
 - /content/drive/MyDrive/KAZ_MORPH/fai